# 01 Clean Donated YouTube Watch Data

This notebook demonstrates the first step in the data-donation workflow: turning raw Google Takeout-style YouTube watch-history files into clean event-level tables.

The notebook uses mock donor data stored in `data/mock_takeout`. The same logic can be adapted to confidential GDPR data donations by changing the input path and adding project-specific consent, exclusion, and participant metadata rules.

## What This Notebook Produces

The notebook writes several CSV files to `outputs/tables`.

- `watch_histories.csv`: all donated watch-history rows combined and standardized.
- `video_histories.csv`: the validated video-watch table used by later notebooks.
- `music_histories.csv`: rows identified as YouTube Music.
- `ad_watch_histories.csv`: rows marked as Google Ads.
- `other_histories.csv`: rows that are neither validated videos, music, nor ads.
- `title_prefix_diagnostics.csv`: counts showing which localized watch-action prefixes were recognized.

The main analysis handoff is `video_histories.csv`, where each row represents one retained YouTube video watch event with a participant ID, timestamp, URL, `video_id`, cleaned title, and channel information when available.

In [ ]:
from pathlib import Path
import re

import pandas as pd

## Locate The Project And Input Data

The notebook is designed to run from the Quarto project root or from the `scripts` folder. The helper below walks upward from the current working directory until it finds the Quarto project and the mock Takeout folder.

In [ ]:
def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        has_quarto_project = (candidate / "_quarto.yml").exists()
        has_mock_data = (candidate / "data" / "mock_takeout").exists()
        if has_quarto_project and has_mock_data:
            return candidate
    raise FileNotFoundError(
        "Could not find the Quarto project root. Run this notebook from "
        "inside youtube_donation_dsa_method."
    )


PROJECT_ROOT = find_project_root()
MOCK_TAKEOUT_DIR = PROJECT_ROOT / "data" / "mock_takeout"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "tables"

WATCH_HISTORIES_PATH = OUTPUT_DIR / "watch_histories.csv"
VIDEO_HISTORIES_PATH = OUTPUT_DIR / "video_histories.csv"
MUSIC_HISTORIES_PATH = OUTPUT_DIR / "music_histories.csv"
AD_WATCH_HISTORIES_PATH = OUTPUT_DIR / "ad_watch_histories.csv"
OTHER_HISTORIES_PATH = OUTPUT_DIR / "other_histories.csv"
TITLE_PREFIX_DIAGNOSTICS_PATH = OUTPUT_DIR / "title_prefix_diagnostics.csv"

print(f"Project folder: {PROJECT_ROOT.name}")
print(f"Mock data folder: {MOCK_TAKEOUT_DIR.relative_to(PROJECT_ROOT).as_posix()}")
print(f"Output folder: {OUTPUT_DIR.relative_to(PROJECT_ROOT).as_posix()}")

## Find Donated Watch-History Files

Google Takeout stores YouTube watch history at `YouTube and YouTube Music/history/watch-history.json`. In this mock dataset, each donor has one folder, so the donor folder name becomes the participant identifier.

In [ ]:
watch_files = sorted(
    MOCK_TAKEOUT_DIR.glob("*/YouTube and YouTube Music/history/watch-history.json")
)

watch_file_index = pd.DataFrame(
    {
        "Participant ID": [path.relative_to(MOCK_TAKEOUT_DIR).parts[0] for path in watch_files],
        "watch_history_file": [path.relative_to(PROJECT_ROOT).as_posix() for path in watch_files],
    }
)

watch_file_index

## Load And Combine Watch Histories

Each JSON file is loaded into a dataframe, annotated with `Participant ID`, and then combined into one raw watch-history table. At this point, the table still contains videos, ads, music rows, and any other Takeout row types that may appear.

In [ ]:
def load_watch_history(path):
    participant_id = path.relative_to(MOCK_TAKEOUT_DIR).parts[0]
    data = pd.read_json(path)
    data["Participant ID"] = participant_id
    return data


watch_histories = pd.concat(
    [load_watch_history(path) for path in watch_files],
    ignore_index=True,
)

watch_histories.shape

## Standardize Columns

The original Takeout fields are useful but not always named for analysis. We rename `title` to `watched_title` and `titleUrl` to `url`, then keep the columns needed for cleaning, auditing, and side-table exports.

In [ ]:
watch_histories = watch_histories.rename(
    columns={
        "title": "watched_title",
        "titleUrl": "url",
    }
)

core_columns = [
    "Participant ID",
    "time",
    "header",
    "watched_title",
    "url",
    "details",
    "subtitles",
]

watch_histories = watch_histories.reindex(columns=core_columns)
watch_histories.head()

## Parse Timestamps

Takeout timestamps are ISO-formatted UTC strings. Some rows include milliseconds and some do not, so we use pandas' ISO8601 parser explicitly.

In [ ]:
watch_histories["time"] = pd.to_datetime(
    watch_histories["time"],
    format="ISO8601",
    utc=True,
)

watch_histories["time"].agg(["min", "max"])

## Extract YouTube Video IDs

The `video_id` is the stable join key for linking donated watch events to scraped or platform-provided metadata. We extract it only from standard YouTube watch URLs with an 11-character ID.

In [ ]:
YOUTUBE_WATCH_URL = re.compile(
    r"^https://www\.youtube\.com/watch\?v=([A-Za-z0-9_-]{11})$"
)


def extract_video_id(url):
    if not isinstance(url, str):
        return pd.NA
    match = YOUTUBE_WATCH_URL.match(url)
    return match.group(1) if match else pd.NA


watch_histories["video_id"] = watch_histories["url"].apply(extract_video_id)
watch_histories["has_valid_video_id"] = watch_histories["video_id"].notna()

watch_histories[["watched_title", "url", "video_id"]].head()

## Handle `details` And Classify Rows

This step follows the validated selection mechanic from the original workflow: a retained video-watch row must have a standard watch URL, an 11-character `video_id`, and no `details` value. Rows with `details` are excluded from `video_histories` because in the original data this field captured ads and other non-standard watch-history records.

We still detect Google Ads explicitly and save them separately, so readers can inspect what was excluded.

In [ ]:
def details_missing(value):
    # This mirrors the original `details.isna()` selection rule.
    # Lists or dictionaries mean Takeout supplied a details field, so the row is not a clean video watch.
    if isinstance(value, (list, dict)):
        return False
    return bool(pd.isna(value))


def has_google_ads_marker(details):
    if not isinstance(details, list):
        return False
    return any(
        isinstance(item, dict) and item.get("name") == "From Google Ads"
        for item in details
    )


watch_histories["details_missing"] = watch_histories["details"].apply(details_missing)
watch_histories["has_google_ads_marker"] = watch_histories["details"].apply(has_google_ads_marker)
watch_histories["is_youtube_music"] = watch_histories["header"].eq("YouTube Music")

video_mask = (
    watch_histories["has_valid_video_id"]
    & watch_histories["details_missing"]
    & ~watch_histories["is_youtube_music"]
)
music_mask = watch_histories["is_youtube_music"]
ad_watch_mask = watch_histories["has_google_ads_marker"]
other_mask = ~(video_mask | music_mask | ad_watch_mask)


def classify_watch_row(row):
    if row["is_youtube_music"]:
        return "youtube_music"
    if row["has_valid_video_id"] and row["details_missing"]:
        return "video_watch"
    if row["has_google_ads_marker"]:
        return "ad_watch"
    if row["has_valid_video_id"]:
        return "video_with_details"
    return "other_watch_row"


watch_histories["watch_row_type"] = watch_histories.apply(classify_watch_row, axis=1)

watch_histories["watch_row_type"].value_counts()

## Extract Channel Fields

Takeout often stores channel information inside the nested `subtitles` field. We extract the first channel name and URL when available. Later notebooks can use these fields to check subscription-linked exposure.

In [ ]:
def first_subtitle_field(subtitles, field):
    if not isinstance(subtitles, list) or not subtitles:
        return pd.NA
    first = subtitles[0]
    if not isinstance(first, dict):
        return pd.NA
    return first.get(field, pd.NA)


def extract_channel_id(channel_url):
    if not isinstance(channel_url, str):
        return pd.NA
    match = re.match(r"^https?://www\.youtube\.com/channel/([A-Za-z0-9_-]+)$", channel_url)
    return match.group(1) if match else pd.NA


watch_histories["channel_title"] = watch_histories["subtitles"].apply(
    lambda subtitles: first_subtitle_field(subtitles, "name")
)
watch_histories["channel_url"] = watch_histories["subtitles"].apply(
    lambda subtitles: first_subtitle_field(subtitles, "url")
)
watch_histories["channel_id"] = watch_histories["channel_url"].apply(extract_channel_id)

watch_histories[["channel_title", "channel_url", "channel_id"]].head()

## Clean Multilingual Watch Titles

Google localizes the action prefix in Takeout titles according to the donor account language. The video title might start with `Watched`, `Så`, `Hai guardato`, `Просмотрено видео`, or another localized phrase.

We therefore keep a small, explicit lookup table of known watch-action prefixes from the original workflow. The notebook removes only known prefixes, records which prefix was removed, and creates a diagnostic for titles where no known prefix was recognized. This avoids silently mangling titles when a new account language appears.

In [ ]:
WATCH_ACTION_PREFIXES = pd.DataFrame(
    [
        {"language_hint": "English", "prefix": "Watched "},
        {"language_hint": "Danish/Norwegian", "prefix": "Så "},
        {"language_hint": "Italian", "prefix": "Hai guardato "},
        {"language_hint": "Russian", "prefix": "Просмотрено видео "},
        {"language_hint": "Polish", "prefix": "Obejrzano: "},
    ]
)

prefix_lookup = dict(
    zip(WATCH_ACTION_PREFIXES["prefix"], WATCH_ACTION_PREFIXES["language_hint"])
)
WATCH_PREFIX_PATTERN = re.compile(
    "^(" + "|".join(re.escape(prefix) for prefix in sorted(prefix_lookup, key=len, reverse=True)) + ")"
)
NON_WORD_SYMBOLS = re.compile(r"[^\w\s\u0400-\u04FF]")

WATCH_ACTION_PREFIXES

In [ ]:
def split_known_watch_prefix(title):
    if not isinstance(title, str):
        return pd.Series(
            {
                "watch_action_prefix": pd.NA,
                "watch_action_language_hint": pd.NA,
                "prefix_removed": False,
                "title_without_watch_prefix": pd.NA,
            }
        )

    match = WATCH_PREFIX_PATTERN.match(title)
    if not match:
        return pd.Series(
            {
                "watch_action_prefix": pd.NA,
                "watch_action_language_hint": pd.NA,
                "prefix_removed": False,
                "title_without_watch_prefix": title,
            }
        )

    prefix = match.group(1)
    return pd.Series(
        {
            "watch_action_prefix": prefix.strip(),
            "watch_action_language_hint": prefix_lookup[prefix],
            "prefix_removed": True,
            "title_without_watch_prefix": title[match.end() :].strip(),
        }
    )


def clean_title_text(title):
    if not isinstance(title, str):
        return pd.NA
    without_symbols = NON_WORD_SYMBOLS.sub("", title)
    return re.sub(r"\s+", " ", without_symbols).strip()


prefix_columns = watch_histories["watched_title"].apply(split_known_watch_prefix)
watch_histories[
    [
        "watch_action_prefix",
        "watch_action_language_hint",
        "prefix_removed",
        "title_without_watch_prefix",
    ]
] = prefix_columns
watch_histories["clean_title"] = watch_histories["title_without_watch_prefix"].apply(clean_title_text)

watch_histories[
    ["watched_title", "watch_action_prefix", "watch_action_language_hint", "clean_title"]
].head()

## Prefix Diagnostics

The prefix diagnostic is a guardrail for real donation projects. If a participant's Takeout export uses an account language that is not in `WATCH_ACTION_PREFIXES`, the title will remain usable, but the action phrase will not be removed. Those cases should be inspected and the prefix list should be extended deliberately.

In [ ]:
prefix_diagnostics = (
    watch_histories.assign(
        watch_action_prefix=watch_histories["watch_action_prefix"].fillna("UNRECOGNIZED"),
        watch_action_language_hint=watch_histories["watch_action_language_hint"].fillna("UNRECOGNIZED"),
    )
    .groupby(["watch_action_prefix", "watch_action_language_hint"], dropna=False)
    .size()
    .reset_index(name="rows")
    .sort_values("rows", ascending=False)
    .reset_index(drop=True)
)

prefix_diagnostics

In [ ]:
unrecognized_prefix_mask = watch_histories["has_valid_video_id"] & ~watch_histories["prefix_removed"]

unrecognized_title_prefix_examples = (
    watch_histories.loc[
        unrecognized_prefix_mask,
        ["Participant ID", "watched_title", "url"],
    ]
    .assign(
        possible_prefix=lambda df: df["watched_title"].str.extract(r"^(\S+(?:\s+\S+){0,2})")[0]
    )
    .head(20)
)

unrecognized_title_prefix_examples

## Create Clean And Side Tables

`video_histories` is the narrow analysis table. It uses the validated rule from above: standard video URL, valid `video_id`, and missing `details`.

The side tables preserve rows that are analytically interesting but excluded from ordinary video exposure: YouTube Music, Google Ads, and other unclassified rows.

In [ ]:
watch_export_columns = [
    "Participant ID",
    "time",
    "header",
    "watch_row_type",
    "watched_title",
    "title_without_watch_prefix",
    "clean_title",
    "watch_action_prefix",
    "watch_action_language_hint",
    "prefix_removed",
    "url",
    "video_id",
    "has_valid_video_id",
    "details_missing",
    "has_google_ads_marker",
    "is_youtube_music",
    "channel_title",
    "channel_url",
    "channel_id",
    "details",
]

video_columns = [
    "Participant ID",
    "time",
    "watched_title",
    "clean_title",
    "watch_action_prefix",
    "watch_action_language_hint",
    "url",
    "video_id",
    "channel_title",
    "channel_url",
    "channel_id",
]

watch_histories_export = (
    watch_histories.loc[:, watch_export_columns]
    .sort_values(["Participant ID", "time"], ascending=[True, False])
    .reset_index(drop=True)
)
video_histories = (
    watch_histories.loc[video_mask, video_columns]
    .sort_values(["Participant ID", "time"], ascending=[True, False])
    .reset_index(drop=True)
)
music_histories = (
    watch_histories.loc[music_mask, watch_export_columns]
    .sort_values(["Participant ID", "time"], ascending=[True, False])
    .reset_index(drop=True)
)
ad_watch_histories = (
    watch_histories.loc[ad_watch_mask, watch_export_columns]
    .sort_values(["Participant ID", "time"], ascending=[True, False])
    .reset_index(drop=True)
)
other_histories = (
    watch_histories.loc[other_mask, watch_export_columns]
    .sort_values(["Participant ID", "time"], ascending=[True, False])
    .reset_index(drop=True)
)

video_histories.head()

## Diagnostics

These checks document what happened during cleaning. For the mock dataset, we expect 300 raw watch-history rows, 270 retained video watches, and 30 ad rows. Music and other rows are included in the workflow even when this mock dataset has none.

In [ ]:
diagnostics = {
    "donor_files_loaded": len(watch_files),
    "raw_watch_rows": len(watch_histories),
    "combined_watch_history_rows": len(watch_histories_export),
    "video_history_rows": len(video_histories),
    "music_history_rows": len(music_histories),
    "ad_watch_history_rows": len(ad_watch_histories),
    "other_history_rows": len(other_histories),
    "invalid_or_missing_video_id_rows": int(watch_histories["video_id"].isna().sum()),
    "unrecognized_video_prefix_rows": int(unrecognized_prefix_mask.sum()),
    "time_min": watch_histories["time"].min(),
    "time_max": watch_histories["time"].max(),
}

pd.Series(diagnostics)

In [ ]:
pd.DataFrame(
    {
        "all_watch_rows": watch_histories.groupby("Participant ID").size(),
        "video_rows": video_histories.groupby("Participant ID").size(),
        "ad_rows": ad_watch_histories.groupby("Participant ID").size(),
    }
).fillna(0).astype(int)

In [ ]:
watch_histories["watch_row_type"].value_counts().rename_axis("watch_row_type").to_frame("rows")

## Save Outputs

The combined `watch_histories.csv` keeps the full cleaned row universe. The narrower `video_histories.csv` is the handoff table for the next step: joining donated watch events to scraped or platform-provided metadata by `video_id`.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

watch_histories_export.to_csv(WATCH_HISTORIES_PATH, index=False)
video_histories.to_csv(VIDEO_HISTORIES_PATH, index=False)
music_histories.to_csv(MUSIC_HISTORIES_PATH, index=False)
ad_watch_histories.to_csv(AD_WATCH_HISTORIES_PATH, index=False)
other_histories.to_csv(OTHER_HISTORIES_PATH, index=False)
prefix_diagnostics.to_csv(TITLE_PREFIX_DIAGNOSTICS_PATH, index=False)

saved_outputs = pd.DataFrame(
    {
        "table": [
            "watch_histories",
            "video_histories",
            "music_histories",
            "ad_watch_histories",
            "other_histories",
            "title_prefix_diagnostics",
        ],
        "path": [
            WATCH_HISTORIES_PATH,
            VIDEO_HISTORIES_PATH,
            MUSIC_HISTORIES_PATH,
            AD_WATCH_HISTORIES_PATH,
            OTHER_HISTORIES_PATH,
            TITLE_PREFIX_DIAGNOSTICS_PATH,
        ],
    }
)
saved_outputs["path"] = saved_outputs["path"].apply(
    lambda output_path: output_path.relative_to(PROJECT_ROOT).as_posix()
)

saved_outputs

## Result

The workflow now has both a full combined watch-history table and a validated video-watch table. The central join key for later enrichment is `video_id`, while the side tables and prefix diagnostics make the cleaning decisions auditable.

### Stored Table Previews

The cells below show the first five rows of each dataframe saved as a CSV. Empty previews are expected when the mock data do not contain that row type.

In [ ]:
stored_watch_tables = {
    "watch_histories.csv": watch_histories_export,
    "video_histories.csv": video_histories,
    "music_histories.csv": music_histories,
    "ad_watch_histories.csv": ad_watch_histories,
    "other_histories.csv": other_histories,
    "title_prefix_diagnostics.csv": prefix_diagnostics,
}

for file_name, dataframe in stored_watch_tables.items():
    print(f"{file_name} ({len(dataframe)} rows)")
    display(dataframe.head(5))